# Docker for ML Deployment — Examples

This notebook introduces container concepts and generates a small Docker-ready Python HTTP application. It does not require Docker to run the Python examples, but the final section gives commands to build and test the container.

**Note:** the example endpoint is a demonstration service, not a production prediction API. The code does not download data or load serialized models.

## 1. Check the Docker CLI (optional)

In [ ]:
import shutil
import subprocess

docker_path = shutil.which("docker")
if docker_path:
    result = subprocess.run(
        ["docker", "--version"],
        capture_output=True,
        text=True,
        check=False,
    )
    print(result.stdout.strip() or result.stderr.strip())
else:
    print("Docker CLI was not found. You can still complete the Python and Dockerfile exercises.")

## 2. Create a minimal health API

In [ ]:
from pathlib import Path

project_dir = Path("docker-ml-demo")
project_dir.mkdir(exist_ok=True)

app_source = r'''
from http.server import BaseHTTPRequestHandler, HTTPServer
import json
import os

PORT = int(os.environ.get("PORT", "8000"))

class Handler(BaseHTTPRequestHandler):
    def do_GET(self):
        if self.path == "/health":
            status, body = 200, {"status": "ok"}
        else:
            status, body = 404, {"error": "not found"}

        payload = json.dumps(body).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json")
        self.send_header("Content-Length", str(len(payload)))
        self.end_headers()
        self.wfile.write(payload)

HTTPServer(("0.0.0.0", PORT), Handler).serve_forever()
'''.lstrip()

(project_dir / "app.py").write_text(app_source, encoding="utf-8")
print((project_dir / "app.py").read_text(encoding="utf-8"))

## 3. Write a Dockerfile and `.dockerignore`

In [ ]:
dockerfile = '''FROM python:3.12-slim
ENV PYTHONDONTWRITEBYTECODE=1 PYTHONUNBUFFERED=1
WORKDIR /app
COPY app.py .
EXPOSE 8000
USER 10001
CMD ["python", "app.py"]
'''

dockerignore = '''.git/
.venv/
__pycache__/
*.py[cod]
.ipynb_checkpoints/
artifacts/
.env
'''

(project_dir / "Dockerfile").write_text(dockerfile, encoding="utf-8")
(project_dir / ".dockerignore").write_text(dockerignore, encoding="utf-8")

print("Dockerfile:\n")
print((project_dir / "Dockerfile").read_text(encoding="utf-8"))
print(".dockerignore:\n")
print((project_dir / ".dockerignore").read_text(encoding="utf-8"))

## 4. Run the application without Docker (optional)

In [ ]:
import subprocess
import sys
import time
import urllib.request
import urllib.error

# This starts the server in the background for a brief local smoke test.
# If port 8000 is already occupied, change PORT below.
env = dict(__import__("os").environ)
env["PORT"] = "8765"
server = subprocess.Popen(
    [sys.executable, str(project_dir / "app.py")],
    env=env,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)

try:
    response = None
    for attempt in range(20):
        try:
            response = urllib.request.urlopen("http://127.0.0.1:8765/health", timeout=1)
            break
        except Exception:
            time.sleep(0.2)
    if response is None:
        print("Could not reach local demo server. Check whether the port is available.")
    else:
        print("HTTP status:", response.status)
        print("Body:", response.read().decode("utf-8"))
finally:
    server.terminate()
    try:
        server.wait(timeout=3)
    except subprocess.TimeoutExpired:
        server.kill()
        server.wait()

## 5. Build and run with Docker

From a terminal, change into the generated project folder and run:

```bash
cd docker-ml-demo
docker build -t health-api:1.0 .
docker run --rm --name health-api -p 8000:8000 health-api:1.0
```

In a second terminal:

```bash
curl http://localhost:8000/health
```

Expected response:

```json
{"status": "ok"}
```

Stop the container with `Ctrl+C`. For troubleshooting, use `docker ps`, `docker logs <container>`, and `docker inspect <container>`.

## 6. Reflection

- An image is a template; a container is an instance of an image.
- `-p 8080:8000` maps host port 8080 to container port 8000.
- The service binds to `0.0.0.0` so it can be reached through the container network.
- Secrets and local environment files should not be copied into the image.
- Production ML deployments should pin dependencies, validate input schemas, run with least privilege, and load only trusted model artifacts.